BPE'yi sıfırdan yaz ve kendi Türkçe metninde eğit. Geçen hafta derlediğin Türkçe metni kullan. Ardışık ikilileri say, en sık geçeni yeni bir token'la birleştir, istediğin vocab boyutuna gelene kadar tekrarla. İlk 20 birleştirmeyi metin olarak yazdır: hangi harf grupları önce birleşiyor? Sıkıştırma oranını (byte sayısı / token sayısı) yaz. (0:23:50 - 0:39:20)

In [1]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
tokens = text.encode("utf-8") # ham byte'lar
tokens = list(map(int, tokens)) # 0..255 arası tam sayılar
print("karakter sayısı:", len(text))
print("byte sayısı:", len(tokens))
print(text[5000:5300])

karakter sayısı: 365585
byte sayısı: 400857
 alafranga olan bir adam artık hanesine Arap 
çorap doldurur mu? Bâ-husûs 25 ki aralıkta bir alafranga dostları dahi gel -
mekte olduğundan bunlar meyânında hizmet etmek için Rum ve Ermeni 
hizmetçilere ihtiyacı der-kârdır26.
Bizim asıl maksadımız Felâtun Bey’i haber vermek yani erbâb-ı 27 
mütâlaay


### Ardışık ikilileri say

In [3]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [4]:
stats = get_stats(tokens)
print("farklı ikili sayısı:", len(stats))
# en sık geçen 10 ikili
for count, pair in sorted(((v, k) for k, v in stats.items()), reverse=True)[:10]:
    print(count, pair, repr(bytes(pair).decode("utf-8", errors="replace")))

farklı ikili sayısı: 1917
11446 (196, 177) 'ı'
5458 (110, 32) 'n '
5317 (101, 32) 'e '
4602 (32, 98) ' b'
4508 (195, 162) 'â'
4448 (105, 32) 'i '
4394 (101, 114) 'er'
4223 (197, 159) 'ş'
4012 (97, 110) 'an'
3955 (97, 32) 'a '


In [5]:
top_pair = max(stats, key=stats.get)
top_pair

(196, 177)

### En sık ikiliyi yeni token'la birleştir

In [6]:
def merge(ids, pair, idx):
    # ids içinde art arda gelen pair'leri yeni idx token'ı ile değiştir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [7]:
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print("önce :", len(tokens))
print("sonra:", len(tokens2))

[5, 6, 99, 9, 1]
önce : 400857
sonra: 389411


### Tokenizer Eğitim döngüsü

In [8]:
vocab_size = 1000 # istediğimiz son vocab boyutu
num_merges = vocab_size - 256
ids = list(tokens) # orijinali bozmamak için kopya

merges = {} # (int, int) -> int
freqs = [] # birleştirme anındaki frekans
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    freqs.append(stats[pair])
    if i % 100 == 0:
        print(f"{i}/{num_merges} birleştirme, token sayısı: {len(ids)}")

0/744 birleştirme, token sayısı: 389411
100/744 birleştirme, token sayısı: 246161
200/744 birleştirme, token sayısı: 211564
300/744 birleştirme, token sayısı: 193089
400/744 birleştirme, token sayısı: 180434
500/744 birleştirme, token sayısı: 170567
600/744 birleştirme, token sayısı: 162911
700/744 birleştirme, token sayısı: 156725


### İlk 20 birleştirme (metin olarak)

In [9]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def show(b):
    return b.decode("utf-8", errors="replace")

for i, ((p0, p1), idx) in enumerate(list(merges.items())[:20]):
    print(f"{idx}: {show(vocab[p0])!r:8} + {show(vocab[p1])!r:8} -> {show(vocab[idx])!r:10} ({freqs[i]} kez)")

256: '�'      + '�'      -> 'ı'        (11446 kez)
257: 'n'      + ' '      -> 'n '       (5458 kez)
258: 'e'      + ' '      -> 'e '       (5317 kez)
259: '�'      + '�'      -> 'â'        (4508 kez)
260: 'i'      + ' '      -> 'i '       (4448 kez)
261: 'e'      + 'r'      -> 'er'       (4394 kez)
262: '�'      + '�'      -> 'ş'        (4223 kez)
263: 'a'      + ' '      -> 'a '       (3955 kez)
264: '�'      + '�'      -> 'ü'        (3853 kez)
265: 'a'      + 'r'      -> 'ar'       (3672 kez)
266: 'i'      + 'r'      -> 'ir'       (2943 kez)
267: 'i'      + 'n'      -> 'in'       (2834 kez)
268: 'a'      + 'n'      -> 'an'       (2636 kez)
269: '�'      + '�'      -> 'ğ'        (2484 kez)
270: 'm'      + 'e'      -> 'me'       (2372 kez)
271: 'd'      + 'i'      -> 'di'       (2306 kez)
272: ','      + ' '      -> ', '       (2269 kez)
273: '�'      + '�'      -> 'ç'        (2209 kez)
274: 'e'      + 'n'      -> 'en'       (2151 kez)
275: 'a'      + 'k'      -> 'ak'       (2096 kez)

In [10]:
# son 10 birleştirme ve en uzun 15 token
for (p0, p1), idx in list(merges.items())[-10:]:
    print(f"{idx}: {show(vocab[p0])!r} + {show(vocab[p1])!r} -> {show(vocab[idx])!r}")
print()
print(sorted([show(vocab[i]) for i in range(256, vocab_size)], key=len, reverse=True)[:15])

990: 'd' + 'et' -> 'det'
991: '2' + '6' -> '26'
992: 'ı' + 't' -> 'ıt'
993: '. ' + 'Z' -> '. Z'
994: 'ka' + 'p' -> 'kap'
995: 't' + 'ü' -> 'tü'
996: 'ç' + 'al' -> 'çal'
997: 'de' + 'ş' -> 'deş'
998: 'c' + 'an' -> 'can'
999: 's' + 'er' -> 'ser'

['\nFelâtun Bey ile Râkım Efendi — ', '\nAnadolu Üniversitesi', 'Anadolu Üniversitesi', ' ile Râkım Efendi — ', ' ile Râkım Efendi �', ' ile Râkım Efendi —', 'Anadolu Üniversite', ' ile Râkım Efendi ', 'Anadolu Üniver', '.\nY ozefino - ', 'Râkım Efendi ', 'Y ozefino - ', '\nFelâtun Bey', 'Felâtun Bey', 'Anadolu Üni']


Felâtun Bey ile Râkım Efendi gibi sonuçların çıkması bu metinler kitaplardan alındığı için sürekli kitabın ismi yazarın ismi gibi çok fazla tekrar eden kelimeleri görüp bunlar sürekli yan yana geliyor o zaman bunları bütün olarak alayım diyor ama bu istediğimiz bir durum değil

### Sıkıştırma oranı

In [11]:
print("byte sayısı :", len(tokens))
print("token sayısı:", len(ids))
print(f"sıkıştırma oranı: {len(tokens) / len(ids):.2f}X")

byte sayısı : 400857
token sayısı: 154399
sıkıştırma oranı: 2.60X
